# Lakehouse Diagnostics

Step-by-step troubleshooting notebook to isolate **where** lakehouse access fails.

Run the cells top-to-bottom. Each section is independent and prints a clear `PASS` / `FAIL`
so you can see exactly at which layer the permission problem starts:

1. Basic execution (does the kernel run at all?)
2. Running identity (who am I executing as?)
3. Default lakehouse / mount info
4. Spark catalog read (list databases & tables)
5. Table read (`SELECT`)
6. File system read (ABFS / Files & Tables paths)
7. Write – create table
8. Write – insert rows
9. Write – file system write
10. Cleanup
11. Summary

## 0. Test harness

A tiny helper that records every check so the final cell can print a summary table.

In [ ]:
import traceback
from datetime import datetime, timezone

RESULTS = []

def check(name):
    """Decorator-style context manager to run a test and record PASS/FAIL."""
    class _Check:
        def __enter__(self):
            print(f"\n=== {name} ===")
            self.start = datetime.now(timezone.utc)
            return self
        def __exit__(self, exc_type, exc, tb):
            ok = exc_type is None
            msg = "" if ok else f"{exc_type.__name__}: {exc}"
            RESULTS.append((name, ok, msg))
            if ok:
                print(f"PASS  {name}")
            else:
                print(f"FAIL  {name}")
                print("      " + "\n      ".join(traceback.format_exception_only(exc_type, exc)).strip().splitlines()[0])
            return True  # swallow exception so the notebook keeps running
    return _Check()

print("Harness ready.")

## 1. Basic execution

Confirms the kernel runs Python and Spark is available — nothing touches the lakehouse yet.

In [ ]:
with check("1. Basic execution / print"):
    print("Hello from the diagnostics notebook.")
    print("Spark version:", spark.version)
    print("Application name:", spark.conf.get("spark.app.name", "<unknown>"))

## 2. Running identity

Prints the security context the notebook executes under. A permission problem in one
environment is almost always because **this identity** differs from the one that has
access to the lakehouse (e.g. a service principal / managed identity vs. a user).

In [ ]:
with check("2. Running identity"):
    from notebookutils import mssparkutils

    # Token-based identity (works for both users and service principals)
    try:
        ctx = mssparkutils.runtime.context
        print("Runtime context:")
        for k, v in ctx.items():
            print(f"   {k}: {v}")
    except Exception as e:
        print("runtime.context unavailable:", e)

    # Decode the AAD token to reveal the actual principal (upn / appid / oid)
    try:
        import base64, json
        token = mssparkutils.credentials.getToken("pbi")
        payload = token.split(".")[1]
        payload += "=" * (-len(payload) % 4)  # pad base64
        claims = json.loads(base64.urlsafe_b64decode(payload))
        print("\nToken claims (identity running this notebook):")
        for k in ("upn", "unique_name", "appid", "app_displayname", "oid", "tid", "idtyp", "name"):
            if k in claims:
                print(f"   {k}: {claims[k]}")
        principal_type = "Service Principal / App" if claims.get("idtyp") == "app" or "appid" in claims and "upn" not in claims else "User"
        print(f"\n   => Principal type: {principal_type}")
    except Exception as e:
        print("Could not decode token:", e)

## 3. Default lakehouse / mount info

Shows which lakehouse is attached and its mount paths. If nothing is attached, the
relative-path reads/writes below will fail regardless of permissions.

In [ ]:
with check("3. Default lakehouse / mounts"):
    from notebookutils import mssparkutils

    try:
        print("Default lakehouse id  :", mssparkutils.lakehouse.getDefault())
    except Exception as e:
        print("getDefault() failed:", e)

    print("\nActive mounts:")
    try:
        for m in mssparkutils.fs.mounts():
            print(f"   mountPoint={m.mountPoint}  source={m.source}")
    except Exception as e:
        print("fs.mounts() failed:", e)

    print("\nCurrent database:", spark.catalog.currentDatabase())

## 4. Spark catalog read (metadata)

Lists databases and tables. This is a **metastore** read — it can succeed even when
the underlying storage is not accessible, which helps narrow the problem down.

In [ ]:
with check("4. Catalog read (list databases & tables)"):
    print("Databases:")
    for db in spark.catalog.listDatabases():
        print("   ", db.name)

    print("\nTables in current database:")
    tables = spark.catalog.listTables()
    if not tables:
        print("   (none)")
    for t in tables:
        print(f"   {t.name}  (type={t.tableType})")

## 5. Table read (`SELECT`)

Actually reads **data** from storage. If catalog read (step 4) passes but this fails,
the identity can see metadata but lacks **storage / OneLake data** permissions.

In [ ]:
with check("5. Table read (SELECT)"):
    tables = spark.catalog.listTables()
    if not tables:
        print("No tables to read — skipping (not a failure).")
    else:
        target = tables[0].name
        print(f"Reading from first table: {target}")
        df = spark.sql(f"SELECT * FROM {target} LIMIT 5")
        cnt = df.count()
        print(f"   Rows fetched (max 5): {cnt}")
        df.show(truncate=False)

## 6. File system read (ABFS / OneLake)

Lists the `Tables/` and `Files/` folders directly via the file system API.
Distinguishes Spark-layer issues from raw OneLake storage permission issues.

In [ ]:
with check("6. File system read (Tables/ & Files/)"):
    from notebookutils import mssparkutils
    for path in ("Tables", "Files"):
        try:
            entries = mssparkutils.fs.ls(path)
            print(f"{path}/  ->  {len(entries)} entries")
            for e in entries[:10]:
                print(f"     {e.name}  (isDir={e.isDir}, size={e.size})")
        except Exception as e:
            print(f"{path}/  ->  ERROR: {e}")
            raise

## 7. Write – create table

First write test. Creates a throwaway diagnostics table. A failure here with reads
passing means the identity has **read-only** access to the lakehouse.

In [ ]:
DIAG_TABLE = "_lakehouse_diag_tmp"

with check("7. Write - CREATE TABLE"):
    spark.sql(f"DROP TABLE IF EXISTS {DIAG_TABLE}")
    spark.sql(f"""
        CREATE TABLE {DIAG_TABLE} (
            id      INT,
            note    STRING,
            created TIMESTAMP
        ) USING DELTA
    """)
    print(f"Created table: {DIAG_TABLE}")

## 8. Write – insert rows

In [ ]:
with check("8. Write - INSERT rows"):
    from datetime import datetime, timezone
    now = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')
    spark.sql(f"""
        INSERT INTO {DIAG_TABLE} VALUES
            (1, 'diagnostic write test', TIMESTAMP '{now}'),
            (2, 'second row',            TIMESTAMP '{now}')
    """)
    back = spark.sql(f"SELECT * FROM {DIAG_TABLE} ORDER BY id")
    print("Rows written and read back:")
    back.show(truncate=False)
    assert back.count() == 2, "expected 2 rows"

## 9. Write – file system write

Writes a small text file under `Files/`. Tests raw OneLake write permission,
independent of the Delta/Spark table path.

In [ ]:
DIAG_FILE = "Files/_lakehouse_diag_tmp.txt"

with check("9. Write - file system write"):
    from notebookutils import mssparkutils
    mssparkutils.fs.put(DIAG_FILE, f"diag write {datetime.now(timezone.utc).isoformat()}", overwrite=True)
    content = mssparkutils.fs.head(DIAG_FILE)
    print("Wrote and read back file content:")
    print("   ", content)

## 10. Cleanup

Removes the throwaway table and file. Cleanup failures are reported but do not
affect the diagnosis.

In [ ]:
with check("10. Cleanup"):
    from notebookutils import mssparkutils
    try:
        spark.sql(f"DROP TABLE IF EXISTS {DIAG_TABLE}")
        print(f"Dropped table {DIAG_TABLE}")
    except Exception as e:
        print("Table cleanup skipped:", e)
    try:
        if mssparkutils.fs.exists(DIAG_FILE):
            mssparkutils.fs.rm(DIAG_FILE)
            print(f"Removed file {DIAG_FILE}")
    except Exception as e:
        print("File cleanup skipped:", e)

## 11. Summary

The table below shows the first failing layer. Typical interpretations:

| First failure | Likely cause |
|---|---|
| 2. Identity | Token/identity not available — check how the notebook is invoked |
| 3. Mounts / 4. Catalog | No lakehouse attached, or no workspace role at all |
| 5. Table read / 6. FS read | Identity has metadata access but **no OneLake data read** permission |
| 7–9. Writes | Identity is **read-only** (Viewer) — needs Contributor/Member or OneLake write |

In [ ]:
print("=" * 70)
print("DIAGNOSTIC SUMMARY")
print("=" * 70)
width = max((len(n) for n, _, _ in RESULTS), default=10)
for name, ok, msg in RESULTS:
    status = "PASS" if ok else "FAIL"
    line = f"  {status}  {name.ljust(width)}"
    if not ok:
        line += f"   -> {msg}"
    print(line)

first_fail = next((n for n, ok, _ in RESULTS if not ok), None)
print("-" * 70)
if first_fail:
    print(f"First failing layer: {first_fail}")
    print("See the interpretation table in the cell above to localize the permission gap.")
else:
    print("All checks passed — this identity has full read/write access to the lakehouse.")

In [ ]:
from notebookutils import mssparkutils

# Get the actual table location (relative paths don't resolve for managed tables)
table_detail = spark.sql("DESCRIBE DETAIL your_table_name").collect()[0]
table_path = table_detail["location"]

# Verify _delta_log exists
try:
    log_files = mssparkutils.fs.ls(f"{table_path}/_delta_log/")
    print(f"Found {len(log_files)} log files")
    display(log_files)
except Exception as e:
    print(f"_delta_log not found: {e}")

# Check if location has any Delta characteristics
from delta.tables import DeltaTable
try:
    dt = DeltaTable.forPath(spark, table_path)
    print("Valid Delta table")
except:
    print("Not a valid Delta table - metadata missing")